# Imports and Spark session


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# Create utilities

In [0]:
dbutils.widgets.text("catalog", "lakehouse_stocks", "Catalog")
dbutils.widgets.dropdown("data_source", "twelvedata", ["twelvedata","fmp"], "Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

In [0]:
S3_BASE = "s3://modern-lakehouse-raw"
landing_path = f"{S3_BASE}/landing_zone/{data_source}/"
checkpoint_path = f"{S3_BASE}/_checkpoints/bronze_{data_source}/"
schema_location = f"{S3_BASE}/_checkpoints/bronze_{data_source}_schema/"
target_table = f"{catalog}.bronze.{data_source}_raw"

In [0]:
print(f"Reading from: {landing_path}")
print(f"Schema Locatin : {schema_location}")
print(f"Checkpoint at: {checkpoint_path}")
print(f"Target table: {target_table}")

# Autoloader Stream Reader & Metadata Enrichment

In [0]:
df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format","json")
    .option("cloudFiles.schemaLocation",schema_location)
    .option("cloudFiles.inferColumnTypes","true")
    .option("cloudFiles.schemaEvolutionMode","addNewColumns")
    .load(landing_path)
    .withColumn("ingested_at", F.current_timestamp())
    .select("*", "_metadata.file_name", "_metadata.file_size")
)

#Executing the Autoloader Write Stream

In [0]:
query = (
    df_stream.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_path)
        .option("delta.enableChangeDataFeed", "true")
        .trigger(availableNow=True)
        .toTable(target_table)
)

query.awaitTermination()

# inspect the table and verify the schema inference

In [0]:
bronze_df = spark.table(target_table)
bronze_df.printSchema()

In [0]:
print(f"Total rows in {target_table}: {bronze_df.count()}")



In [0]:
display(bronze_df.limit(10))
